## Разведка API маркетплейса

**Цель:** понять структуру данных, которые отдаёт API `final-project.simulative.ru`

In [2]:
# Подключаем библиотеки для HTTP-запросов и работы с JSON
import requests
import json

In [3]:
# Отправляем запрос к API за 1 января 2023, таймаут 60 сек
url = "http://final-project.simulative.ru/data?date=2023-01-01"
r = requests.get(url, timeout=60)

# смотрим, что вернулось
print(r.status_code)
print(len(r.text))
print(r.headers.get("Content-Type"))

200
1907566
text/html; charset=utf-8


In [4]:
# Парсим json и смотрим сколько записей и какой тип
data = r.json()
print(type(data)) 
print(len(data))

<class 'list'>
8092


In [5]:
# Смотрим первые две записи в JSON
print(json.dumps(data[:2], ensure_ascii=False, indent=2))

[
  {
    "client_id": 373032,
    "gender": "F",
    "purchase_datetime": "2023-01-01",
    "purchase_time_as_seconds_from_midnight": 55382,
    "product_id": 30569,
    "quantity": 66,
    "price_per_item": 72492,
    "discount_per_item": 26530,
    "total_price": 3033492.0
  },
  {
    "client_id": 658080,
    "gender": "M",
    "purchase_datetime": "2023-01-01",
    "purchase_time_as_seconds_from_midnight": 26969,
    "product_id": 31098,
    "quantity": 0,
    "price_per_item": 2436,
    "discount_per_item": 1150,
    "total_price": 0.0
  }
]


In [6]:
# Ключевые метрики для проектирования БД
print(f"Записей: {len(data)}")
print(f"Уникальных client_id: {len(set(r['client_id'] for r in data))}")
print(f"Уникальных product_id: {len(set(r['product_id'] for r in data))}")
print(f"Gender: {set(r['gender'] for r in data)}")
print(f"Даты в данных: {sorted(set(r['purchase_datetime'] for r in data))}")

# Аномалии — нужно ли CHECK-констрейнты
print(f"quantity=0: {sum(1 for r in data if r['quantity'] == 0)}")
print(f"quantity<0: {sum(1 for r in data if r['quantity'] < 0)}")
print(f"price_per_item<0: {sum(1 for r in data if r['price_per_item'] < 0)}")
print(f"discount>price: {sum(1 for r in data if r['discount_per_item'] > r['price_per_item'])}")

Записей: 8092
Уникальных client_id: 8060
Уникальных product_id: 7495
Gender: {'M', 'F'}
Даты в данных: ['2023-01-01']
quantity=0: 160
quantity<0: 0
price_per_item<0: 0
discount>price: 0


In [8]:
# Проверим на дубли
from collections import Counter

# Собираем ключи из 4 полей — то, что могло бы быть уникальным
keys = []
for r in data:
    key = (
        r["client_id"],
        r["product_id"],
        r["purchase_datetime"],
        r["purchase_time_as_seconds_from_midnight"]
    )
    keys.append(key)

# Считаем частоты и отбираем те, что встретились больше раза
counter = Counter(keys)
dupes = [(k, v) for k, v in counter.items() if v > 1]

print(f"Всего записей: {len(keys)}")
print(f"Уникальных ключей: {len(counter)}")
print(f"Дублей: {len(dupes)}")

dupes[:3]

Всего записей: 8092
Уникальных ключей: 8092
Дублей: 0


[]

In [38]:
# Проверяем, с какой даты API отдаёт данные — нужно для backfill
test_dates = ["2023-06-15", "2022-01-01", "2021-01-01", "2020-01-01"]

for test_date in test_dates:
    r_test = requests.get(f"http://final-project.simulative.ru/data?date={test_date}", timeout=60)
    try:
        d = r_test.json()
        print(f"{test_date}: {len(d)} записей")
    except Exception as e:
        print(f"{test_date}: ошибка — {e}")

2023-06-15: 1992 записей
2022-01-01: 7461 записей
2021-01-01: ошибка — Expecting value: line 1 column 1 (char 0)
2020-01-01: ошибка — Expecting value: line 1 column 1 (char 0)


#### Выводы:

API отдаёт данные за один день по запросу `?date=YYYY-MM-DD`. Формат — JSON-массив, ~8 000 записей за день (~1.9 МБ).

Поля в записи:

- `client_id` — id клиента
- `gender` — пол (M/F)
- `purchase_datetime` — дата покупки (только дата)
- `purchase_time_as_seconds_from_midnight` — время в секундах от полуночи
- `product_id` — id товара
- `quantity` — количество
- `price_per_item` — цена за штуку (копейки)
- `discount_per_item` — скидка за штуку (копейки)
- `total_price` — итог (копейки)

Что выяснил:

- **Уникального id в данных нет.** Пришлось собрать композитный ключ из 4 полей — `(client_id, product_id, purchase_datetime, purchase_time_as_seconds_from_midnight)`. Проверил на 8 092 записях — дублей 0.
- **API отдаёт с 2022-01-01.** Раньше — пустой ответ.
- **Аномалий нет:** отрицательных quantity, price, discount нет; скидка не превышает цену. Только `quantity = 0` — 160 записей (~2%).
- **Деньги — в копейках.** Медиана чека 388 717 (~3 887 руб.), максимум 7 722 714 (~77 227 руб.).

Что это значит для БД:

- Таблица `raw.sales` — 9 полей + `id` + `loaded_at`.
- `UNIQUE` по 4 полям композитного ключа.
- `CHECK` на gender, quantity, price, discount.
- Индексы: `client_id`, `product_id`, `purchase_datetime`.